# 6.03 Semantic Chunker

`langchain_experimental.text_splitter.SemanticChunker` 는 **문장 단위 임베딩 거리** 를 계산해 **의미 경계** 에서 자르는 splitter 다. 문자 수 / 토큰 수가 아니라 **임베딩 코사인 거리** 가 기준.

## 학습 목표

- 동작 원리(인접 문장 임베딩 거리 → breakpoint 탐지) 를 이해한다
- `breakpoint_threshold_type` 4종(`percentile` / `standard_deviation` / `interquartile` / `gradient`) 을 구분한다
- `buffer_size`, `number_of_chunks`, `min_chunk_size` 등 보조 파라미터의 의미를 안다
- RAG 에서 Recursive vs Semantic 분할 결과를 비교한다
- 비용 · 지연 · 품질 트레이드오프를 판단할 수 있다

## 언제 쓰나

- **문단 구분이 불명확한 긴 글**(음성 전사, 회의록, 인터뷰 녹취) — 헤더/줄바꿈 splitter 무력
- **주제 전환이 중요한 문서** — 한 chunk 안에 여러 토픽이 섞이면 retrieval 정확도 ↓
- **chunk 개수 목표** 가 있을 때(`number_of_chunks=N`)

**트레이드오프** — 분할 과정에서 문장마다 임베딩 호출이 필요 → 비용·지연 증가. 짧은 문서(< 1k 문장) 에 적합.

## 6.03.1 환경 설정

필요 패키지: `langchain-experimental`, `langchain-openai`. `.env` 에 `OPENAI_API_KEY` 필요.
설치 안 되어 있으면: `uv pip install langchain-experimental`

In [ ]:
# !pip install -U langchain-experimental langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Splitter 03 는 OPENAI_API_KEY 필요 (문장 임베딩 호출)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
print('import OK')

## 6.03.2 기본 사용법

알고리즘:
1. 입력 텍스트를 `sentence_split_regex`(기본 `(?<=[.?!])\s+`) 로 문장 분리
2. 각 문장을 앞뒤 `buffer_size`(기본 1) 이웃과 합쳐 임베딩
3. 인접 벡터 간 **코사인 거리** 계산
4. `breakpoint_threshold_type` 기준으로 **이상치 거리** 를 가진 경계에서 분리

시그니처:
```python
SemanticChunker(
    embeddings: Embeddings,
    buffer_size: int = 1,
    add_start_index: bool = False,
    breakpoint_threshold_type: Literal['percentile', 'standard_deviation', 'interquartile', 'gradient'] = 'percentile',
    breakpoint_threshold_amount: float | None = None,
    number_of_chunks: int | None = None,
    sentence_split_regex: str = r'(?<=[.?!])\s+',
    min_chunk_size: int | None = None,
)
```

In [ ]:
SAMPLE = (
    "LangChain is a framework for building LLM applications. "
    "It offers document loaders, embeddings, vector stores, and agents. "
    "RAG (Retrieval-Augmented Generation) is a core pattern for reducing hallucination. "
    "The quality of retrieval depends heavily on how documents are chunked. "
    "Now shifting topic — Korean BBQ has a distinct regional variety. "
    "Seoul-style focuses on marinated short ribs known as galbi. "
    "Jeonju-style bibimbap is a famous non-BBQ Korean dish. "
    "Finally, let's talk about weather. "
    "Seoul has four distinct seasons with hot humid summers and cold dry winters. "
    "Busan is milder year-round due to its coastal location."
)

emb = OpenAIEmbeddings(model="text-embedding-3-small")

chunker = SemanticChunker(
    embeddings=emb,
    breakpoint_threshold_type="percentile",   # 기본값
    breakpoint_threshold_amount=95,            # 상위 5% 거리에서 분리
)
chunks = chunker.split_text(SAMPLE)
print(f'chunks: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'[{i}] {c[:80]}…')

## 6.03.3 `breakpoint_threshold_type` 4종 비교

| 타입 | 기본 amount | 의미 | 성향 |
|------|-------------|------|------|
| `percentile` | 95 | 거리의 상위 N% 를 경계로 | 기본값. 분포에 robust |
| `standard_deviation` | 3 | 평균 + Nσ 이상 | 거리 분포가 **정규에 가까울 때** |
| `interquartile` | 1.5 | Q3 + N·IQR 이상 | **이상치 탐지 방식**. 왜곡 분포에 강함 |
| `gradient` | 95 | 거리의 기울기 상위 N% | **급격한 주제 전환** 에 유리 |

→ 동일 문서에 4 타입을 돌려 chunk 수를 비교한다.

In [ ]:
results = {}
for t in ["percentile", "standard_deviation", "interquartile", "gradient"]:
    ch = SemanticChunker(embeddings=emb, breakpoint_threshold_type=t)
    results[t] = ch.split_text(SAMPLE)

print(f'{"type":<22} | {"#chunks":>7} | {"avg 문자":>10}')
print('-'*50)
for t, chs in results.items():
    avg = sum(len(c) for c in chs) // max(len(chs), 1)
    print(f'{t:<22} | {len(chs):>7} | {avg:>10}')

## 6.03.4 보조 파라미터

- `buffer_size`(기본 1) — 문장 하나를 앞뒤 N 문장과 합쳐 임베딩. 크면 매끄러운 경계, 작으면 예민
- `number_of_chunks` — **원하는 chunk 수 고정**. 이 값이 주어지면 threshold 를 자동 역산
- `min_chunk_size` — 너무 짧은 조각을 다음 chunk 와 병합
- `sentence_split_regex` — 기본 영문(`.?!`). **한국어** 는 `(?<=[.?!。\n])\s+` 처럼 확장 고려
- `add_start_index` — `Document` metadata 에 원문 기준 start offset 기록

In [ ]:
fixed = SemanticChunker(
    embeddings=emb,
    number_of_chunks=3,          # 정확히 3덩이
    buffer_size=1,
    min_chunk_size=50,
    add_start_index=True,
)
docs = fixed.create_documents([SAMPLE])
print(f'docs: {len(docs)}')
for d in docs:
    print(d.metadata, '|', d.page_content[:60], '…')

## 6.03.5 RAG 비교 — Recursive vs Semantic

같은 문서로 두 splitter 결과를 같은 vector store 에 넣고, **주제 전환이 걸친 쿼리** 로 검색 품질을 비교한다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

rec = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
rec_chunks = rec.split_text(SAMPLE)

sem = SemanticChunker(embeddings=emb, breakpoint_threshold_type="percentile")
sem_chunks = sem.split_text(SAMPLE)

def build_store(chunks, tag):
    docs = [Document(page_content=c, metadata={"splitter": tag, "i": i}) for i, c in enumerate(chunks)]
    return InMemoryVectorStore.from_documents(docs, embedding=emb)

rec_store = build_store(rec_chunks, "recursive")
sem_store = build_store(sem_chunks, "semantic")

query = "What is Korean BBQ like?"
print('--- recursive top-1 ---')
print(rec_store.similarity_search(query, k=1)[0].page_content[:120])
print('\n--- semantic  top-1 ---')
print(sem_store.similarity_search(query, k=1)[0].page_content[:120])

## 6.03.6 비용 · 지연 · 품질 트레이드오프

| 항목 | RecursiveCharacter | Semantic |
|------|--------------------|----------|
| 전처리 비용 | **무료** | 문장당 임베딩 호출 1회 |
| 지연 | ms 단위 | 네트워크 I/O + 임베딩 계산 |
| 결과 결정성 | 결정적 | 결정적(같은 임베더) |
| 주제 전환 처리 | ✗ | **✓** |
| 구조적 경계(헤더) | ✗ | ✗ (Markdown splitter 와 체인) |
| 규모 적합성 | 수백만 문서 | 수십~수천 문서 |

### 하이브리드 전략
1. `MarkdownHeaderTextSplitter` 로 **구조 경계** 분리
2. 큰 섹션은 `SemanticChunker` 로 **의미 경계** 분할
3. 여전히 큰 조각만 `RecursiveCharacterTextSplitter` 로 **크기 한도** 적용

## 체크리스트

- [ ] 비용 민감 대량 문서엔 Recursive, 품질 민감 소규모 문서엔 Semantic
- [ ] threshold 타입은 문서 분포 (균일 vs 왜곡) 에 맞춰 선택
- [ ] 한국어면 `sentence_split_regex` 확장 필요
- [ ] `number_of_chunks` 로 chunk 수를 상수화하면 인덱스 예산 관리가 쉬움

## 다음

- `../03_vectorstores/` — 분할된 chunk 를 vector store 에 싣기
- `../04_retrievers/` — 분할 전략 + retriever 조합

## 참고

- 소스: https://github.com/langchain-ai/langchain-experimental/blob/main/libs/experimental/langchain_experimental/text_splitter.py
- 원 아이디어(Greg Kamradt): https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb
- 공식 가이드: https://python.langchain.com/docs/how_to/semantic-chunker/